# UAV Object Detection — Baseline (Colab GPU)

**YOLO11s + P2 başlığı**, tam kare fine-tune (dilimleme YOK — o ayrı bir deneme).

Bu skor sonraki denemelerin referans noktası.

**Sıra:** Runtime > Change runtime type > GPU. Sonra hücreleri sırayla çalıştır.

## 1. Depo + Drive
Depoyu ve veriyi Drive'a koyup buradan aç.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/proje && unzip -q -o /content/drive/MyDrive/uav-object-detection.zip -d /content/proje
%cd /content/proje
!pip install -q -e .

## 2. Bağımlılıklar + ortam doğrulama

In [ ]:
!pip install -q -r requirements-train.txt
import torch, ultralytics
print('ultralytics', ultralytics.__version__)
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 3. Veri
`data/` klasörünü Drive'dan kopyala. Beklenen: `data/uav_ldz/{train,val,test}/images` + `data/coco/instances_*.json`.

`configs/data.yaml` `path:` satırını Colab yoluna göre yaz.

In [ ]:
!cp -r /content/drive/MyDrive/uav_data/* data/

cfg = '''path: /content/proje/data/uav_ldz
train: train/images
val: val/images
test: test/images
names:
  0: vehicle
  1: human
  2: uap
  3: uai
'''
open('configs/data.yaml', 'w').write(cfg)

!ls data/uav_ldz/*/images | head && echo '---' && ls data/coco

## 4. Eğitim
`configs/baseline.yaml`: yolo11s+P2, imgsz 1280, 150 epoch, batch 32, COCO ağırlığından fine-tune.
`runs/baseline/baseline-yolo11s-p2/` altına yazar.

In [ ]:
!python scripts/train.py --config configs/baseline.yaml

## 5. Referans skor (kendi eval aracı)
Ultralytics val hızlı kontrol; kesin skor = `predict_to_coco.py` -> `evaluate.py` (mAP@0.5, sınıf-başı AP, faster-coco-eval çapraz).

In [ ]:
W = 'runs/baseline/baseline-yolo11s-p2/weights/best.pt'

!python scripts/predict_to_coco.py $W test --out preds_test.json
!python scripts/evaluate.py preds_test.json data/coco/instances_test.json --cross-check --json score_test.json

## 6. Çıktıları Drive'a kaydet

In [ ]:
!mkdir -p /content/drive/MyDrive/uav_runs/baseline
!cp -r runs/baseline/baseline-yolo11s-p2 /content/drive/MyDrive/uav_runs/baseline/
!cp score_test.json /content/drive/MyDrive/uav_runs/baseline/